**EXTRACTING FEATURES FOR EACH OF TRAINING, VALIDATION and TEST SUBJECTS (for each slice) FOR PASSING TO KNN CLASSIFIER**


In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model

# 1. Re-define the custom loss
def semi_hard_triplet_loss(labels, embeddings):
    return tf.reduce_mean(tf.square(embeddings))

# 2. Path to your model
model_path = './model_checkpoints/my_model.h5'

# 3. Load with safe_mode=False to allow the Lambda layer
model = load_model(
    model_path, 
    custom_objects={
        'semi_hard_triplet_loss': semi_hard_triplet_loss,
        'tf': tf  # <--- THIS FIXES THE NAMEERROR
    },
    safe_mode=False 
)

print("Model loaded successfully!")

Model loaded successfully!


In [2]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from skimage.transform import resize

# 1. Path to your images
datapath_pr = 'image_dataset/train'

# 2. Function to scan folders instead of using a .pkl file
def get_images_from_folders(base_path):
    image_paths = []
    labels = []
    # Assumes your folders are named 'normal' and 'parkinson'
    class_names = ['normal', 'parkinson'] 
    
    for label_idx, cls in enumerate(class_names):
        cls_path = os.path.join(base_path, cls)
        if os.path.exists(cls_path):
            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.png', '.jpg', '.npy')):
                    image_paths.append(os.path.join(cls_path, img_name))
                    labels.append(label_idx)
    return image_paths, labels

# 3. Load the file paths
all_img_paths, all_labels = get_images_from_folders(datapath_pr)

# 4. Load and process images for the Model/KNN
processed_images = []

for path in all_img_paths:
    if path.lower().endswith('.npy'):
        img_array = np.load(path, allow_pickle=True)
    else:
        # Load .png or .jpg and convert to numpy array
        img = image.load_img(path) 
        img_array = image.img_to_array(img)

    # Resize to exactly 121x121 and normalize
    img_array = resize(img_array, (121, 121)) / 255.0
    processed_images.append(img_array)
    
arr_tr = np.array(processed_images)
print(f"Total images loaded: {len(arr_tr)}")
print(f"Array shape: {arr_tr.shape}")

Total images loaded: 581
Array shape: (581, 121, 121, 3)


In [3]:
import numpy as np

all_features = []
# Ensure this list is the same length as arr_tr
# Example: y_labels = [0, 0, 1, 1...] based on your folders
y_train_knn = np.array(all_labels) 

for i in range(len(arr_tr)):
    # 1. Get the image directly (it is already 121x121x3)
    img = arr_tr[i]

    # 2. Add batch dimension to make it (1, 121, 121, 3)
    img_batch = np.expand_dims(img, axis=0)
    
    # 3. Predict
    prediction = model.predict(img_batch, verbose=0)
    
    # Get the 64-dim embedding
    all_features.append(prediction[0].flatten())

X_train_knn = np.array(all_features)
print(f"Extraction complete. Features shape: {X_train_knn.shape}")

Extraction complete. Features shape: (581, 64)


In [4]:
# 1. DEFINE the path
adi_path = 'pd_features'

# 2. CREATE the folder (This fixes the OSError)
if not os.path.exists(adi_path):
    os.makedirs(adi_path)
    print(f"Created directory: {adi_path}")

# 3. Save Feature Vectors
df_features = pd.DataFrame(X_train_knn)
df_features.to_csv(os.path.join(adi_path, "feature_train.csv"), index=True)

# 4. Save Labels
df_labels = pd.DataFrame(y_train_knn, columns=['Label'])
df_labels.to_csv(os.path.join(adi_path, "labels_AD1_CN0_train.csv"), index=True)

# 5. Save Image Names
filenames = [os.path.basename(p) for p in all_img_paths]
df_names = pd.DataFrame(filenames, columns=['filename'])
df_names.to_csv(os.path.join(adi_path, "features_train_slice_name.csv"), index=False)

print(f"Success! All files saved in the '{adi_path}' folder.")

Success! All files saved in the 'pd_features' folder.


In [5]:
#VALIDATING
import os
import numpy as np
from tensorflow.keras.preprocessing import image

# 1. Point to your VALIDATION folder
val_datapath = 'image_dataset/val' 
val_class_names = ['normal', 'parkinson'] 

val_img_paths = []
val_labels = []

# 2. Scan Validation Folders
for label_idx, cls in enumerate(val_class_names):
    cls_path = os.path.join(val_datapath, cls)
    if os.path.exists(cls_path):
        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(('.png', '.jpg', '.npy')):
                val_img_paths.append(os.path.join(cls_path, img_name))
                val_labels.append(label_idx)

# 3. Predict/Extract
val_features = []
print(f"Extracting features for {len(val_img_paths)} validation images...")

for path in val_img_paths:
    if path.lower().endswith('.npy'):
        img_array = np.load(path, allow_pickle=True)
    else:
        img = image.load_img(path)
        img_array = image.img_to_array(img)

    img_array = resize(img_array, (121, 121)) / 255.0
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Use the same model to get embeddings
    prediction = model.predict(img_batch, verbose=0)
    val_features.append(prediction[0].flatten())

X_val_knn = np.array(val_features)
y_val_knn = np.array(val_labels)

print(f"Validation Extraction Complete! Shape: {X_val_knn.shape}")


Extracting features for 124 validation images...
Validation Extraction Complete! Shape: (124, 64)


In [6]:
import pandas as pd

# 1. Save Validation Features
df_val_features = pd.DataFrame(X_val_knn)
df_val_features.to_csv(os.path.join(adi_path, "feature_test.csv"), index=True)

# 2. Save Validation Labels
df_val_labels = pd.DataFrame(y_val_knn, columns=['Label'])
df_val_labels.to_csv(os.path.join(adi_path, "labels_AD1_CN0_test.csv"), index=True)

# 3. Save Validation Image Names
val_filenames = [os.path.basename(p) for p in val_img_paths]
df_val_names = pd.DataFrame(val_filenames, columns=['filename'])
df_val_names.to_csv(os.path.join(adi_path, "features_test_slice_name.csv"), index=False)

print(f"All Validation CSV files saved in: {adi_path}")

All Validation CSV files saved in: pd_features


In [7]:
#testING

# 1. Point to your TEST folder
test_datapath = 'image_dataset/test' 
test_class_names = ['normal', 'parkinson'] 

test_img_paths = []
test_labels = []

# 2. Scan Test Folders
for label_idx, cls in enumerate(test_class_names):
    cls_path = os.path.join(test_datapath, cls)
    if os.path.exists(cls_path):
        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(('.png', '.jpg', '.npy')):
                test_img_paths.append(os.path.join(cls_path, img_name))
                test_labels.append(label_idx)

# 3. Predict/Extract Features for Test Set
test_features = []
print(f"Extracting features for {len(test_img_paths)} test images...")

for path in test_img_paths:
    if path.lower().endswith('.npy'):
        img_data = np.load(path, allow_pickle=True)
        # If it's a 3D volume, take the middle slice. If 2D, use as is.
        if len(img_data.shape) == 3:
            mid = img_data.shape[0] // 2
            slice_2d = img_data[mid]
        else:
            slice_2d = img_data
    else:
        # Load standard images (.png, .jpg)
        img = image.load_img(path)
        slice_2d = image.img_to_array(img)
    slice_resized = resize(slice_2d, (121, 121), anti_aliasing=True)
    
    # Ensure it has 3 channels (RGB)
    if len(slice_resized.shape) == 2:  # If grayscale (121, 121)
        img_rgb = np.stack([slice_resized] * 3, axis=-1)
    elif slice_resized.shape[-1] == 4: # If RGBA, remove transparency
        img_rgb = slice_resized[:, :, :3]
    else:
        img_rgb = slice_resized

    img_batch = np.expand_dims(img_rgb, axis=0)
    prediction = model.predict(img_batch, verbose=0)
    test_features.append(prediction[0].flatten())

X_test_knn = np.array(test_features)
y_test_knn = np.array(test_labels)

print(f"Test Extraction Complete! Shape: {X_test_knn.shape}")

Extracting features for 127 test images...
Test Extraction Complete! Shape: (127, 64)


In [8]:
# 1. Save Test Features
df_test_features = pd.DataFrame(X_test_knn)
df_test_features.to_csv(os.path.join(adi_path, "feature_final_test.csv"), index=True)

# 2. Save Test Labels
df_test_labels = pd.DataFrame(y_test_knn, columns=['Label'])
df_test_labels.to_csv(os.path.join(adi_path, "labels_AD1_CN0_final_test.csv"), index=True)

# 3. Save Test Image Names
test_filenames = [os.path.basename(p) for p in test_img_paths]
df_test_names = pd.DataFrame(test_filenames, columns=['filename'])
df_test_names.to_csv(os.path.join(adi_path, "features_final_test_slice_name.csv"), index=False)

print(f"All Final Test CSV files saved in: {adi_path}")

All Final Test CSV files saved in: pd_features


In [9]:
from sklearn.neighbors import KNeighborsClassifier

# 1. Initialize the KNN (5 neighbors is standard)
knn = KNeighborsClassifier(n_neighbors=5)

# 2. Train it using the features (X) and labels (y) you extracted earlier
# This assumes you have already run the 'Extraction' cells for the training set
knn.fit(X_train_knn, y_train_knn)

print("KNN Classifier is now trained and ready!")

KNN Classifier is now trained and ready!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Use the KNN to predict labels for the test features
y_pred = knn.predict(X_test_knn)

# 2. Compare predicted labels to the actual true labels
accuracy = accuracy_score(y_test_knn, y_pred)

print(f"--- Model Accuracy ---")
print(f"Total Accuracy: {accuracy * 100:.2f}%")
print("\n--- Detailed Report ---")
print(classification_report(y_test_knn, y_pred, target_names=['Normal', 'Parkinson']))

--- Model Accuracy ---
Total Accuracy: 98.43%

--- Detailed Report ---
              precision    recall  f1-score   support

      Normal       0.98      1.00      0.99        92
   Parkinson       1.00      0.94      0.97        35

    accuracy                           0.98       127
   macro avg       0.99      0.97      0.98       127
weighted avg       0.98      0.98      0.98       127



In [11]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Predict on Training Data
y_train_pred = knn.predict(X_train_knn)
train_acc = accuracy_score(y_train_knn, y_train_pred)

# 2. Predict on Test Data (New Patients)
y_test_pred = knn.predict(X_test_knn)
test_acc = accuracy_score(y_test_knn, y_test_pred)

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Test Accuracy: {test_acc * 100:.2f}%")



Training Accuracy: 100.00%
Test Accuracy: 98.43%


In [12]:
model.save("models/feature_extractor.h5")